In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
import os
from collections import Counter
import re

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

## 1. Offline NER Database Statistics

Load and aggregate data from `searching_results/offline_ner_db/`.

In [ ]:
ner_db_path = "../searching_results/offline_ner_db/"
json_files = glob.glob(os.path.join(ner_db_path, "ner_results_batch_*.json"))

print(f"Found {len(json_files)} batch files.")

entity_counts = Counter()
entity_names = {}

for file_path in json_files:
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
            
        for file_result in data.get('per_file_results', []):
            entities = file_result.get('entities', {})
            for entity_id, info in entities.items():
                mentions = info.get('mentions', [])
                count = len(mentions)
                entity_counts[entity_id] += count
                
                # Try to extract a name if we don't have one yet
                if entity_id not in entity_names and mentions:
                    # Extract name from "name (Context: ...)"
                    first_mention = mentions[0]
                    match = re.match(r"^(.*?)\s*\(Context:", first_mention)
                    if match:
                        entity_names[entity_id] = match.group(1).strip()
                    else:
                        entity_names[entity_id] = first_mention[:50] # Fallback
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"Total unique entities: {len(entity_counts)}")
print(f"Total mentions: {sum(entity_counts.values())}")

In [ ]:
# Convert to DataFrame for plotting
df_ner = pd.DataFrame.from_dict(entity_counts, orient='index', columns=['count'])
df_ner['name'] = df_ner.index.map(lambda x: entity_names.get(x, x))
df_ner = df_ner.sort_values('count', ascending=False)

print(df_ner.head(10))

In [ ]:
# Plot Top 20 Entities
plt.figure(figsize=(12, 8))
sns.barplot(data=df_ner.head(20), x='count', y='name', palette='viridis')
plt.title('Top 20 Most Frequent Chemical Entities in Offline Database')
plt.xlabel('Mention Count')
plt.ylabel('Chemical Entity')
plt.tight_layout()
plt.show()

## 2. Agent Performance Analysis

Load `searching_results/batch_summary.json`.

In [ ]:
batch_summary_path = "../searching_results/batch_summary.json"

if os.path.exists(batch_summary_path):
    with open(batch_summary_path, 'r') as f:
        batch_data = json.load(f)
    
    df_batch = pd.DataFrame(batch_data)
    print(f"Loaded {len(df_batch)} records from batch summary.")
    print(df_batch.head())
else:
    print("Batch summary file not found.")
    df_batch = pd.DataFrame()

In [ ]:
if not df_batch.empty:
    # Plot Claim Distribution
    plt.figure(figsize=(8, 6))
    claim_counts = df_batch['claim'].value_counts()
    plt.pie(claim_counts, labels=claim_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title('Distribution of Agent Claims')
    plt.show()

    # Plot Confidence Score Distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(df_batch['confidence_score'], bins=20, kde=True)
    plt.title('Distribution of Confidence Scores')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.show()

    # Confidence vs Claim
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df_batch, x='claim', y='confidence_score')
    plt.title('Confidence Score by Claim Type')
    plt.show()

## 3. Comparison with Ground Truth (if available)

Load `notebooks/metabolites_endo_exo_llm_gemini2.5.csv`.

In [ ]:
csv_path = "metabolites_endo_exo_llm_gemini2.5.csv"
if os.path.exists(csv_path):
    df_comp = pd.read_csv(csv_path)
    print(f"Loaded {len(df_comp)} records from comparison CSV.")
    
    # Confusion Matrix Plot
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    # Filter for valid labels if necessary
    valid_labels = ['Endogenous', 'Exogenous']
    df_filtered = df_comp[df_comp['Origin'].isin(valid_labels) & df_comp['Endogenous_LLM'].isin(valid_labels)]
    
    if not df_filtered.empty:
        cm = confusion_matrix(df_filtered['Origin'], df_filtered['Endogenous_LLM'], labels=valid_labels)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=valid_labels)
        
        plt.figure(figsize=(8, 8))
        disp.plot(cmap='Blues', values_format='d')
        plt.title('Confusion Matrix: Ground Truth vs Agent')
        plt.grid(False)
        plt.show()
    else:
        print("Not enough matching data for confusion matrix.")
else:
    print("Comparison CSV not found.")

## 4. Single Run Analysis (Online Search & NER)

Analyze the results from a specific run (e.g., `run_20251212_190649`) to demonstrate the pipeline's capability to retrieve and extract information without the full offline database.

In [ ]:
import networkx as nx
from itertools import combinations

# Path to a specific run's NER results
# Example 1: Manual run
# run_id = "run_20251212_190649"
# run_ner_path = f"../searching_results/{run_id}/ner_results/chemical_ner_results.json"

# Example 2: E2E run (check for literature_searching subfolder)
run_id = "e2e_run_20251210_105347"
run_ner_path = f"../searching_results/{run_id}/literature_searching/ner_results/chemical_ner_results.json"

if os.path.exists(run_ner_path):
    with open(run_ner_path, 'r') as f:
        run_data = json.load(f)
    
    print(f"Loaded NER results for run {run_id}")
    print(f"Total mentions: {run_data.get('mention_count', 0)}")
    
    # Extract entities per file
    file_entities = {}
    all_entities = Counter()
    entity_names_run = {}

    for file_result in run_data.get('per_file_results', []):
        file_path = file_result.get('file')
        entities = file_result.get('entities', {})
        
        current_file_entities = []
        for entity_id, info in entities.items():
            mentions = info.get('mentions', [])
            count = len(mentions)
            all_entities[entity_id] += count
            current_file_entities.append(entity_id)
            
            # Extract name
            if entity_id not in entity_names_run and mentions:
                first_mention = mentions[0]
                # Try to parse "Name (Context: ...)"
                match = re.match(r"^(.*?)\s*\(Context:", first_mention)
                if match:
                    entity_names_run[entity_id] = match.group(1).strip()
                else:
                    entity_names_run[entity_id] = first_mention.split('(')[0].strip()[:30]
        
        file_entities[file_path] = current_file_entities

    # Top Entities Plot
    if all_entities:
        df_run_ner = pd.DataFrame.from_dict(all_entities, orient='index', columns=['count'])
        df_run_ner['name'] = df_run_ner.index.map(lambda x: entity_names_run.get(x, x))
        df_run_ner = df_run_ner.sort_values('count', ascending=False)
        
        plt.figure(figsize=(12, 6))
        sns.barplot(data=df_run_ner.head(15), x='count', y='name', palette='magma')
        plt.title(f'Top 15 Chemicals Found in Run {run_id}')
        plt.xlabel('Mention Count')
        plt.ylabel('Chemical Entity')
        plt.tight_layout()
        plt.show()
    else:
        print("No entities found in this run.")

else:
    print(f"Run results not found at {run_ner_path}")

In [ ]:
# Co-occurrence Network
if os.path.exists(run_ner_path) and len(file_entities) > 0:
    G = nx.Graph()
    
    # Add edges for chemicals appearing in the same paper
    for f_path, ents in file_entities.items():
        if len(ents) > 1:
            for e1, e2 in combinations(ents, 2):
                if G.has_edge(e1, e2):
                    G[e1][e2]['weight'] += 1
                else:
                    G.add_edge(e1, e2, weight=1)
    
    # Filter nodes/edges to keep the graph readable
    # Keep edges with weight > 1 or top nodes
    
    # Let's keep only the top 20 most frequent entities to visualize their connections
    top_entities = df_run_ner.head(20).index.tolist()
    subgraph = G.subgraph(top_entities)
    
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(subgraph, k=0.5, iterations=50)
    
    # Node sizes based on count
    node_sizes = [all_entities[n] * 50 for n in subgraph.nodes()]
    
    # Labels
    labels = {n: entity_names_run.get(n, n) for n in subgraph.nodes()}
    
    nx.draw_networkx_nodes(subgraph, pos, node_size=node_sizes, node_color='skyblue', alpha=0.7)
    nx.draw_networkx_edges(subgraph, pos, width=1.0, alpha=0.5)
    nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=10)
    
    plt.title(f'Chemical Co-occurrence Network (Top 20 Entities) - Run {run_id}')
    plt.axis('off')
    plt.show()
else:
    print("Skipping network plot.")